# Diviation Check
Figure out how the time avergaed gradient differs from the normal gradient.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
LR = 0.1  # learning rate

X = np.array([
    [0, 0],
    [0, 1],
    [1, 0],
    [1, 1]
]) * 0.8  # XOR inputs

Y = np.array([
    [0], [1], [1], [0]]) * 0.8  # XOR outputs

print("X:", X)
print("Y:", Y)


# time averaged init

In [ ]:
adj = np.array(
    [
        # x1, x2, h1, h2, o1
        [1, 1, 0, 0, 0], # h1
        [1, 1, 0, 0, 0], # h2
        [0, 0, 1, 1, 0] # o1
    ]
)

output_mask = np.array([0, 0, 1])  # only the output neuron

w = np.array([
    [0.25914638, 0.77686861, 0.         , 0.         , 0.        ],
    [0.5485403 , 0.94806484, 0.         , 0.         , 0.        ],
    [0.         , 0.         , 0.27204435, 0.10031312, 0.        ]
])
# w = w * adj  # apply adjacency mask

b = np.array([0.26781006, 0.94080729, 0.56385632]) # biases between -1 and 1

print("Weights:\n", w)
print("Biases:\n", b)

state = np.repeat([ 0, 0, 0], len(X)).reshape(len(X), -1)
print("Initial state:", state)

def activation(x):
    return (x > 0).astype(float)

def activation_prime(x):
    return np.where(x > 0, 1, 0)

def noisy_iterate(state, X, w, b):
    noise = np.random.rand(*X.shape)
    x = (noise < X).astype(float)
    state = np.concatenate((x, state), axis=1)
    a = state @ w.T + b
    state = activation(a)
    output = state[:, -1]
    return a, state, output


In [ ]:
yHats = []
states = []
aes = []
for i in range(500):
    a, state, yHat = noisy_iterate(state, X, w, b)
    states.append(state)
    yHats.append(yHat)
    aes.append(a)


yHat = np.mean(yHats, axis=0).reshape(-1, 1)
avg_state = np.mean(states, axis=0)
avg_a = np.mean(aes, axis=0)
print("Avg a:", avg_a, avg_a.shape)
print("Average state:", avg_state, avg_state.shape)
print("yHat:", yHat, yHat.shape)
print("Y:", Y, Y.shape)

In [ ]:
# ---------------- Backward Initialization as function -------------------
def backward_init(yHat, Y, state, output_idx):
    # loss gradient
    dJ = yHat-Y # yHat(txo) - y (txo) = (4x1)
    # setup initial delta state
    delta_state = np.zeros_like(state) # (5x4)
    delta_state[output_idx, :] = dJ.T # (5x4)[1:4] <= (1x4)

    return delta_state

# ---------------- backward step as function -------------------
def backward_step(delta_state, nr_inputs, states, z_states, w, b, adj):
    delta = delta_state[nr_inputs:] * activation_prime(z_states)
    dw = (delta @ states.T) * adj
    return w.T @ delta, dw

delta_state = backward_init(yHat, Y, state, output_mask.astype(bool))
print("delta_state:", delta_state, delta_state.shape)
for i in range(2):  # two backward steps
    delta_state, dw = backward_step(delta_state, len(X), state, a, w, b, adj)
    print(f"After step {i+1}:")
    print("delta_state:", delta_state, delta_state.shape)
    print("dw:", dw, dw.shape)
    w -= LR * dw
    print("Updated weights:\n", w)

# Standard FFN

In [ ]:
# ---------- Aktivierungsfunktionen ----------
def relu(z):
    return np.maximum(0, z)

def relu_prime(z):
    return (z > 0).astype(float)

# ---------- Initialisierung ----------

W1 = np.array([
    [0.25914638, 0.5485403 ],   # Gewichte von x1 -> h1,h2
    [0.77686861, 0.94806484],   # Gewichte von x2 -> h1,h2
])
b1 = np.array([[0.26781006, 0.94080729]])  # h1,h2

# Hidden -> Output (2x1)
W2 = np.array([
    [0.27204435],   # h1 -> o1
    [0.10031312],   # h2 -> o1
])
b2 = np.array([[0.56385632]])  # o1

LR = 0.1

# ----- Forward -----
z1 = X @ W1 + b1     # (4,2)
print("z1", z1)
a1 = relu(z1)        # (4,2)
print("a1", a1)

z2 = a1 @ W2 + b2    # (4,1)
print("z2", z2)
yhat = relu(z2)      # (4,1)
print("yhat", yhat)

# Loss (MSE)
loss = np.mean((yhat - Y) ** 2)
print("Loss:", loss)

# ----- Backward -----
# Output-Delta
dJ = (yhat - Y)                # (4,1)
print("dJ:", dJ)
delta2 = dJ * relu_prime(z2)   # (4,1)
print("delta2:", delta2)
print("reshaped delta2:", delta2)

dW2 = a1.T @ delta2            # (2,1)
print("dW2:", dW2)
db2 = np.sum(delta2, axis=0, keepdims=True)

# Hidden-Delta
delta1 = (delta2 @ W2.T) * relu_prime(z1)  # (4,2)
print("delta1:", delta1)
dW1 = X.T @ delta1                         # (2,2)
print("dW1:", dW1)
db1 = np.sum(delta1, axis=0, keepdims=True)
